In [18]:
import pandas as pd

df = pd.read_csv(
    "C:/Users/MSI/mini_projects/project_2/data/raw/food_business_information_raw.csv",
    escapechar='\\',
    engine='python'
)

# treat whitespace-only/empty strings as true NaN
df['location_gps'] = df['location_gps'].replace(r'^\s*,?\s*$', pd.NA, regex=True)

coords = df['location_gps'].str.split(',', expand=True)
df['latitude'] = pd.to_numeric(coords[0], errors='coerce')
df['longitude'] = pd.to_numeric(coords[1], errors='coerce')

# null out-of-bounds Dubai coordinates
DUBAI_LAT_RANGE = (24.5, 25.7)
DUBAI_LON_RANGE = (54.8, 55.8)
out_of_bounds = (
    (df['latitude'] < DUBAI_LAT_RANGE[0]) | (df['latitude'] > DUBAI_LAT_RANGE[1]) |
    (df['longitude'] < DUBAI_LON_RANGE[0]) | (df['longitude'] > DUBAI_LON_RANGE[1])
)
df.loc[out_of_bounds, ['latitude', 'longitude']] = None

print(f"Total rows: {len(df)}")
print(f"Valid coordinates: {df['latitude'].notna().sum()}")
print(f"Missing/invalid coordinates: {df['latitude'].isna().sum()}")

df = df.drop(columns=['location_gps'])

Total rows: 62997
Valid coordinates: 53733
Missing/invalid coordinates: 9264


In [19]:
print(df.duplicated().sum(), "fully duplicate rows")
df.tail(5)  # peek at the end in case something got appended oddly

27551 fully duplicate rows


,business_activities,cuisine_name,license_no,licensing_agency,number_of_employees,load_timestamp,latitude,longitude
62992,Restaurant,Indian cuisine,1140094,Department of Economic Development- Dubai,21.0,2026-06-29T16:24:56.000Z,25.287654,55.392077
62993,Global Village,Not Applicable,680396,Department of Economic Development- Dubai,1.0,2026-06-29T16:24:56.000Z,25.074094,55.298309
62994,Dubai SME Projects,American Cuisine,927873,Department of Economic Development- Dubai,12.0,2026-06-29T16:24:56.000Z,25.238368,55.455050
62995,General Trading,Not Applicable,734332,Department of Economic Development- Dubai,1.0,2026-06-29T16:24:56.000Z,25.243846,55.310350
62996,Restaurant,Not Applicable,771895,Department of Economic Development- Dubai,1.0,2026-06-29T16:24:56.000Z,25.204929,55.270840


In [20]:
# Are these truly identical rows, or just identical except timestamp?
print(df.duplicated(subset=[c for c in df.columns if c != 'load_timestamp']).sum(), 
      "duplicates ignoring load_timestamp")

# Check license_no specifically - a business should have a unique license
print(df['license_no'].duplicated().sum(), "duplicate license_no rows")
print(df['license_no'].nunique(), "unique licenses out of", len(df), "rows")

27551 duplicates ignoring load_timestamp
37877 duplicate license_no rows
25120 unique licenses out of 62997 rows


In [21]:
df = df.sort_values('load_timestamp')
df_clean = df.drop_duplicates(subset='license_no', keep='last')

print(f"Before: {len(df)} rows")
print(f"After: {len(df_clean)} rows")
print(f"Unique licenses: {df_clean['license_no'].nunique()}")

Before: 62997 rows
After: 25120 rows
Unique licenses: 25120


In [22]:
print(df_clean.isna().sum())
print(f"\nValid coordinates: {df_clean['latitude'].notna().sum()} / {len(df_clean)} ({df_clean['latitude'].notna().sum()/len(df_clean)*100:.1f}%)")

business_activities       0
cuisine_name            900
license_no                0
licensing_agency          0
number_of_employees       0
load_timestamp            0
latitude               3090
longitude              3090
dtype: int64

Valid coordinates: 22030 / 25120 (87.7%)


In [23]:
missing_gps = df_clean['latitude'].isna()

# Does missing GPS correlate with business type?
print(df_clean.loc[missing_gps, 'business_activities'].value_counts().head(15))
print("\n---vs overall---\n")
print(df_clean['business_activities'].value_counts().head(15))

business_activities
Restaurant                                                 1093
Coffee Shop                                                 308
Cafeteria                                                   299
Grocery Store                                               152
Supermarket                                                 134
General Trading                                             131
Food & Beverages Trading (Sub-activity of Trading)          111
Malls/SME/Intaleq                                           103
Food & Beverages Trading (Sub-activity of Food Service)      65
Vegetable and Fruit Trading                                  54
Catering Services                                            49
Mini Store                                                   45
Food Kiosk                                                   33
Dry Food Trading                                             25
Confectionary & Chocolate Trading                            20
Name: count, dtype: 

In [24]:
# Add an explicit flag so downstream analysis can filter cleanly
df_clean['has_valid_gps'] = df_clean['latitude'].notna()

print(f"Rows with valid GPS: {df_clean['has_valid_gps'].sum()} ({df_clean['has_valid_gps'].mean()*100:.1f}%)")

Rows with valid GPS: 22030 (87.7%)


C:\Users\MSI\AppData\Local\Temp\ipykernel_6148\1546482707.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['has_valid_gps'] = df_clean['latitude'].notna()


In [25]:
df_clean = df.drop_duplicates(subset='license_no', keep='last').copy()

In [26]:
df_clean['load_timestamp'] = pd.to_datetime(df_clean['load_timestamp'])

df_clean['year'] = df_clean['load_timestamp'].dt.year
df_clean['month'] = df_clean['load_timestamp'].dt.month

print(df_clean[['load_timestamp', 'year', 'month']].head())
print("\nYear distribution:\n", df_clean['year'].value_counts().sort_index())
print("\nMonth distribution:\n", df_clean['month'].value_counts().sort_index())

                 load_timestamp  year  month
0     2026-06-29 16:24:56+00:00  2026      6
41990 2026-06-29 16:24:56+00:00  2026      6
42000 2026-06-29 16:24:56+00:00  2026      6
41989 2026-06-29 16:24:56+00:00  2026      6
42004 2026-06-29 16:24:56+00:00  2026      6

Year distribution:
 year
2026    25120
Name: count, dtype: int64

Month distribution:
 month
6    25120
Name: count, dtype: int64


In [27]:
df_clean['month_name'] = df_clean['load_timestamp'].dt.month_name()

In [28]:
df_clean = df_clean.drop(columns=['load_timestamp', 'year', 'month'])
# also drop month_name if you added it
print(df_clean.columns.tolist())

['business_activities', 'cuisine_name', 'license_no', 'licensing_agency', 'number_of_employees', 'latitude', 'longitude', 'month_name']


In [29]:
df_clean=df_clean.drop(columns=['month_name'], errors='ignore')

In [30]:
print(df_clean.columns.tolist())

['business_activities', 'cuisine_name', 'license_no', 'licensing_agency', 'number_of_employees', 'latitude', 'longitude']


In [ ]:
df_clean.to_csv(
    "C:/Users/MSI/mini_projects/project_2/data/cleaned/food_business_information_clean.csv",)